In [0]:
-- Q1: discount depth by category
CREATE OR REPLACE TABLE workspace.default.gold_discount_by_category AS
SELECT category,
       COUNT(*)                       AS products,
       ROUND(AVG(mrp), 2)             AS avg_mrp,
       ROUND(AVG(discount_pct), 1)    AS avg_discount_pct
FROM workspace.default.silver_products
GROUP BY category
HAVING COUNT(*) >= 10;

-- Q2: top brands (known brands with at least 5 products)
CREATE OR REPLACE TABLE workspace.default.gold_top_brands AS
SELECT brand,
       COUNT(*)                       AS products,
       ROUND(AVG(discount_pct), 1)    AS avg_discount_pct,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS product_rank
FROM workspace.default.silver_products
WHERE brand IS NOT NULL
GROUP BY brand
HAVING COUNT(*) >= 5;

-- Q3: products per price band, per category
CREATE OR REPLACE TABLE workspace.default.gold_price_bands AS
SELECT category,
       CASE WHEN mrp < 100  THEN '1. Under 100'
            WHEN mrp < 500  THEN '2. 100-499'
            WHEN mrp < 1000 THEN '3. 500-999'
            ELSE                 '4. 1000+' END AS price_band,
       COUNT(*) AS products
FROM workspace.default.silver_products
GROUP BY 1, 2;

-- Q4: price per 100 g, by sub-category (weight-based products only)
CREATE OR REPLACE TABLE workspace.default.gold_price_per_100g AS
SELECT sub_category,
       COUNT(*)                                          AS products,
       ROUND(AVG(selling_price / weight_g * 100), 2)     AS avg_price_per_100g
FROM workspace.default.silver_products
WHERE weight_g IS NOT NULL
GROUP BY sub_category
HAVING COUNT(*) >= 5;

SELECT 'category total' AS check_name, SUM(products) AS n FROM workspace.default.gold_discount_by_category
UNION ALL
SELECT 'price band total', SUM(products) FROM workspace.default.gold_price_bands
UNION ALL
SELECT 'silver rows', COUNT(*) FROM workspace.default.silver_products;


SELECT category, products, avg_discount_pct
FROM workspace.default.gold_discount_by_category
ORDER BY avg_discount_pct DESC;

SELECT COUNT(*) AS dropped_rows
FROM workspace.default.silver_products
WHERE category IS NULL
   OR category NOT IN (SELECT category FROM workspace.default.gold_discount_by_category);